In [1]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO

In [2]:
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore")
    
logging.getLogger().setLevel(logging.ERROR)
    
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)

In [3]:
load_dotenv()
rag_directory = os.getenv('DIRECTORY', 'Text_Document')

In [4]:
def load_documents(directory):
    loader = DirectoryLoader(directory)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", "។", "?", "!", ""]
    )
    docs = text_splitter.split_documents(documents)
    return docs

In [5]:
load_documents(rag_directory)

[Document(metadata={'source': 'Text_Document\\credit_risk.txt'}, page_content='CREDIT RISK KNOWLEDGE BASE Loan Risk Assessment — Questions, Explanations, and Guidance\n\nTOPIC 1: WHAT IS A LOAN GRADE? A loan grade is a letter-based rating assigned to a borrower or a loan application that summarizes the overall credit risk of that borrower. It is determined by the lender based on multiple financial factors including credit score, income, debt-to-income ratio, employment history, and past repayment behavior.\n\nThe grades range from A to G: - Grade A: Excellent creditworthiness. Very low risk of default. - Grade B: Good creditworthiness. Low risk. - Grade C: Fair creditworthiness. Moderate risk. - Grade D: Below average creditworthiness. Above-average risk. - Grade E: Poor creditworthiness. High risk. - Grade F: Very poor creditworthiness. Very high risk. - Grade G: Extremely poor creditworthiness. The highest risk category.\n\nThe grade is one of the most important factors in a loan dec

In [6]:
len(load_documents(rag_directory))

111

In [7]:
class E5Embeddings(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        texts = [f"passage: {t}" for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        return super().embed_query(f"query: {text}")

In [8]:
def main():
    docs = load_documents(rag_directory)

    with redirect_stdout(StringIO()), redirect_stderr(StringIO()):
        embedding = E5Embeddings(
            model_name="intfloat/multilingual-e5-base",
            model_kwargs={"device": "cpu"},        # "cuda" if available
            encode_kwargs={"normalize_embeddings": True}
        )

        Chroma.from_documents(
            documents=docs,
            embedding=embedding,    
            persist_directory="./chroma_db",
            collection_name="credit_risk_corpus",
            collection_metadata={"hnsw:space": "cosine"}  # cosine for normalized vectors
        )

if __name__ == "__main__":
    main()
